## 📊 Stock Tracker

This notebook provides comprehensive analysis of your stock portfolio with:
- **Offline data** from local CSV file
- **Moving averages**: 21DMA, 50DMA, 100DMA, 200DMA
- **Interactive charts** with Plotly
- **Bullish/Bearish signals**
- **Bulk analysis** capabilities

## 📦 Setup & Installation
Import required libraries and load historical data from CSV:

In [3]:
# Install required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

# Interactive plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print("✅ Libraries imported successfully!")
print("📁 Reading data from local CSV file...")

✅ Libraries imported successfully!
📁 Reading data from local CSV file...


### 🔧 Stock Tracker Class Definition
Load historical data from CSV and perform technical analysis:

In [4]:
class StockTrackerOffline:
    """Stock tracker using offline CSV data from 5Paisa"""

    def __init__(self, csv_path):
        """
        Initialize tracker with CSV data

        Args:
            csv_path: Path to nifty.csv file with columns: datetime,open,high,low,close,volume,code
        """
        self.csv_path = csv_path
        self.indicator_options = {21: '21DMA', 50: '50DMA', 100: '100DMA', 200: '200DMA'}

        # Default watchlist from 5Paisa
        self.stock_codes = ['11536', '1594', '1660', '3506', '3787', '7229', '10604', '16669', '157', '2885']

        # Load all data from CSV
        print(f"📁 Loading data from: {csv_path}")
        self.data_all = self._load_csv_data()

        if self.data_all is not None:
            available_codes = self.data_all['code'].unique()
            print(f"✅ Loaded {len(self.data_all)} records for {len(available_codes)} stocks")
            print(f"📊 Stock codes available: {sorted(available_codes)}")
        else:
            print("❌ Failed to load data")

    def _load_csv_data(self):
        """Load and prepare CSV data"""
        try:
            # Read CSV
            df = pd.read_csv(self.csv_path)

            # Normalize column names
            df.columns = df.columns.str.lower()

            # Convert datetime and set as index
            df['datetime'] = pd.to_datetime(df['datetime'])

            # Ensure code is string
            df['code'] = df['code'].astype(str)

            # Rename 'close' to 'Close' for compatibility with plotting
            df = df.rename(columns={'close': 'Close', 'open': 'Open', 'high': 'High', 'low': 'Low', 'volume': 'Volume'})

            return df

        except FileNotFoundError:
            print(f"❌ File not found: {self.csv_path}")
            return None
        except Exception as e:
            print(f"❌ Error loading CSV: {str(e)}")
            return None

    def get_stock_data(self, stock_code):
        """Get data for a specific stock code"""
        if self.data_all is None:
            print("❌ No data loaded")
            return None

        stock_code = str(stock_code)

        # Filter data for this stock
        stock_data = self.data_all[self.data_all['code'] == stock_code].copy()

        if stock_data.empty:
            print(f"❌ No data found for stock code: {stock_code}")
            available = sorted(self.data_all['code'].unique())
            print(f"💡 Available codes: {available}")
            return None

        # Set datetime as index
        stock_data = stock_data.set_index('datetime').sort_index()

        print(f"✅ Loaded {len(stock_data)} days of data for code {stock_code}")
        print(f"   Date range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")

        return stock_data

    def calculate_moving_averages(self, data):
        """Calculate all moving averages"""
        for period, name in self.indicator_options.items():
            data[name] = data['Close'].rolling(window=period).mean()
        return data

    def create_interactive_chart(self, data, stock_code, indicator_period, days=30):
        """Create interactive Plotly chart"""
        last_days = data.tail(days)
        indicator_name = self.indicator_options[indicator_period]

        fig = go.Figure()

        # Add closing price
        fig.add_trace(go.Scatter(
            x=last_days.index,
            y=last_days['Close'],
            mode='lines',
            name=f'Stock {stock_code} Close',
            line=dict(color='#2E86AB', width=3)
        ))

        # Add moving average
        fig.add_trace(go.Scatter(
            x=last_days.index,
            y=last_days[indicator_name],
            mode='lines',
            name=f'{indicator_name}',
            line=dict(color='#F18F01', width=3, dash='dash')
        ))

        # Update layout
        fig.update_layout(
            title=dict(
                text=f'📈 Stock Code {stock_code} - Price vs {indicator_name} (Last {days} Days)',
                x=0.5,
                font=dict(size=20)
            ),
            xaxis_title='Date',
            yaxis_title='Price (₹)',
            hovermode='x unified',
            template='plotly_white',
            height=600,
            width=800,
            showlegend=True,
            legend=dict(x=0.02, y=0.98)
        )

        # Add latest values annotation
        latest_close = last_days['Close'].iloc[-1]
        latest_ma = last_days[indicator_name].iloc[-1]

        fig.add_annotation(
            text=f'<b>Latest Close:</b> ₹{latest_close:.2f}<br><b>Latest {indicator_name}:</b> ₹{latest_ma:.2f}',
            xref="paper", yref="paper",
            x=0.98, y=0.02,
            showarrow=False,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=12)
        )

        fig.show()
        return fig

    def analyze_stock(self, stock_code, indicator_period, show_chart=True, chart_days=30):
        """Analyze a single stock with specified indicator"""
        stock_code = str(stock_code)
        indicator_name = self.indicator_options[indicator_period]

        # Fetch and process data
        data = self.get_stock_data(stock_code)
        if data is None:
            return None

        data = self.calculate_moving_averages(data)

        # Get latest values
        latest_close = data['Close'].iloc[-1]
        latest_indicator = data[indicator_name].iloc[-1]
        latest_date = data.index[-1].strftime('%Y-%m-%d')

        # Calculate percentage difference
        pct_diff = ((latest_close - latest_indicator) / latest_indicator) * 100

        # Display results with rich formatting
        print("\n" + "="*70)
        print(f"                    📊 ANALYSIS RESULTS")
        print("="*70)
        print(f"📈 Stock Code: {stock_code}")
        print(f"📅 Date: {latest_date}")
        print(f"💰 Latest Closing Price: ₹{latest_close:,.2f}")
        print(f"📊 Latest {indicator_name}: ₹{latest_indicator:,.2f}")

        if pct_diff > 0:
            print(f"📈 Price is {pct_diff:.2f}% ABOVE the {indicator_name} - 🟢 BULLISH")
        else:
            print(f"📉 Price is {abs(pct_diff):.2f}% BELOW the {indicator_name} - 🔴 BEARISH")

        print("="*70)

        # Show chart
        if show_chart:
            self.create_interactive_chart(data, stock_code, indicator_period, chart_days)

        return {
            'stock_code': stock_code, 'date': latest_date, 'close': latest_close,
            'indicator_value': latest_indicator, 'indicator_name': indicator_name,
            'pct_difference': pct_diff, 'signal': 'BULLISH' if pct_diff > 0 else 'BEARISH',
            'data': data
        }

    def bulk_analysis(self, stock_list, indicator_period=50):
        """Analyze multiple stocks and return summary DataFrame"""
        print(f"🔍 Analyzing {len(stock_list)} stocks with {self.indicator_options[indicator_period]}...")

        results = []
        for i, stock_code in enumerate(stock_list, 1):
            print(f"\n[{i}/{len(stock_list)}] Stock Code: {stock_code}")
            analysis = self.analyze_stock(stock_code, indicator_period, show_chart=False)

            if analysis:
                results.append({
                    'Stock Code': analysis['stock_code'],
                    'Close (₹)': f"₹{analysis['close']:,.2f}",
                    f'{analysis["indicator_name"]} (₹)': f"₹{analysis['indicator_value']:,.2f}",
                    'Difference (%)': f"{analysis['pct_difference']:+.2f}%",
                    'Signal': analysis['signal']
                })

        if results:
            df = pd.DataFrame(results)

            print(f"\n📊 BULK ANALYSIS SUMMARY ({self.indicator_options[indicator_period]})")
            print("="*90)
            print(df.to_string(index=False))

            return df
        return None


## Add data

In [5]:
# Initialize tracker with CSV path
csv_file_path = '/Users/samdanibasha/Library/CloudStorage/GoogleDrive-ssamdani@gmail.com/My Drive/5Paisa/nifty/nifty.csv'

tracker = StockTrackerOffline(csv_file_path)

print("📊 Ready to analyze your portfolio stocks")

📁 Loading data from: /Users/samdanibasha/Library/CloudStorage/GoogleDrive-ssamdani@gmail.com/My Drive/5Paisa/nifty/nifty.csv
✅ Loaded 2510 records for 10 stocks
📊 Stock codes available: ['10604', '11536', '157', '1594', '1660', '16669', '2885', '3506', '3787', '7229']
📊 Ready to analyze your portfolio stocks


## Execution

### Example 1: Analyze Single Stock

In [6]:
# Analyze a single stock from your watchlist with 50-day moving average
# Available codes: 11536 (TCS), 1594 (INFOSYS), 1660 (ITC), 3506 (titan), 3787 (wipro), 7229 (HCL), 10604 (airtel), 16669 (bajaj auto), 157 (apollo hospital), 2885 (reliance)

analysis = tracker.analyze_stock('11536', 50, show_chart=True, chart_days=30)

# Try other stock codes from your watchlist
# Try other indicators: 21, 50, 100, 200

✅ Loaded 251 days of data for code 11536
   Date range: 2025-01-08 to 2026-01-09

                    📊 ANALYSIS RESULTS
📈 Stock Code: 11536
📅 Date: 2026-01-09
💰 Latest Closing Price: ₹3,212.60
📊 Latest 50DMA: ₹3,172.56
📈 Price is 1.26% ABOVE the 50DMA - 🟢 BULLISH


### Example 2: Bulk Analysis of Top Stocks

In [7]:
# Analyze all stocks from default watchlist
watchlist_stocks = ['11536', '1594', '1660', '3506', '3787', '7229', '10604', '16669', '157', '2885']

df_results = tracker.bulk_analysis(watchlist_stocks, indicator_period=50)

# Show additional statistics
if df_results is not None:
    bullish_count = len(df_results[df_results['Signal'] == 'BULLISH'])
    bearish_count = len(df_results[df_results['Signal'] == 'BEARISH'])

    print(f"\n📈 Bullish stocks: {bullish_count}")
    print(f"📉 Bearish stocks: {bearish_count}")
    print(f"🎯 Bullish ratio: {(bullish_count/len(watchlist_stocks))*100:.1f}%")

🔍 Analyzing 10 stocks with 50DMA...

[1/10] Stock Code: 11536
✅ Loaded 251 days of data for code 11536
   Date range: 2025-01-08 to 2026-01-09

                    📊 ANALYSIS RESULTS
📈 Stock Code: 11536
📅 Date: 2026-01-09
💰 Latest Closing Price: ₹3,212.60
📊 Latest 50DMA: ₹3,172.56
📈 Price is 1.26% ABOVE the 50DMA - 🟢 BULLISH

[2/10] Stock Code: 1594
✅ Loaded 251 days of data for code 1594
   Date range: 2025-01-08 to 2026-01-09

                    📊 ANALYSIS RESULTS
📈 Stock Code: 1594
📅 Date: 2026-01-09
💰 Latest Closing Price: ₹1,616.90
📊 Latest 50DMA: ₹1,575.71
📈 Price is 2.61% ABOVE the 50DMA - 🟢 BULLISH

[3/10] Stock Code: 1660
✅ Loaded 251 days of data for code 1660
   Date range: 2025-01-08 to 2026-01-09

                    📊 ANALYSIS RESULTS
📈 Stock Code: 1660
📅 Date: 2026-01-09
💰 Latest Closing Price: ₹338.25
📊 Latest 50DMA: ₹396.67
📉 Price is 14.73% BELOW the 50DMA - 🔴 BEARISH

[4/10] Stock Code: 3506
✅ Loaded 251 days of data for code 3506
   Date range: 2025-01-08 to 2026-0

## 💡 Tips & Usage Guide

### How to Use This Notebook:

1. **📦 Setup**: Run the import cell to load libraries
2. **📊 Initialize**: Load CSV data and create tracker instance
3. **🎯 Analyze**: Use the examples or create custom analyses

### Understanding Signals:

- **🟢 BULLISH**: Stock price is above the moving average (potential uptrend)
- **🔴 BEARISH**: Stock price is below the moving average (potential downtrend)

### Moving Average Periods:

- **21DMA**: Short-term trend (3 weeks)
- **50DMA**: Medium-term trend (2.5 months)
- **100DMA**: Long-term trend (5 months)
- **200DMA**: Very long-term trend (10 months)

### Stock Codes:
**Default Watchlist**: 11536, 1594, 1660, 3506, 3787, 7229, 10604, 16669, 157, 2885

### Data Source:
- **CSV File**: `/nifty/nifty.csv`
- **Columns**: datetime, open, high, low, close, volume, code